In [ ]:
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision.transforms import Compose, Resize, ToTensor, Lambda, Grayscale

# 图片的预处理，输入为PIL.Image格式
# GrayScale 灰度图，通道数仅为1(相对地，RGB图片通道数为3，带透明通道的RGBA图片通道数为4)
# Resize 改变图片分辨率
# 将PIL.Image转化为张量(Tensor)，灰度图的取值范围[0, 255]也在这一步变为[0, 1]，数值类型从整数变为浮点数
# 将取值从[0, 1]转为[-1, 1]，又被称为规则化，当数据为规则化后的数据时，神经网络往往才能在质量、速度等方面获得显著提升(详细原因暂不展开)
transform = Compose([Grayscale(), Resize(28), ToTensor(), Lambda(lambda t: (t * 2) - 1)])
# 将 MNIST_PNG/train 下的内容作为训练用数据集
train_data = datasets.ImageFolder('MNIST_PNG/train', transform=transform)
# 将 MNIST_PNG/test 下的内容作为测试用数据集
test_data = datasets.ImageFolder('MNIST_PNG/test', transform=transform)
# 创建数据集读取器
# batch_size: 训练时候每一次训练的图片数量，若内存不够，应当适量减小该值，当使用GPU加速时，应关注显存而非内存
# shuffle: 训练时打乱图片顺序
bs = 128
train_loader = DataLoader(train_data, batch_size=bs, shuffle=True)
test_loader = DataLoader(test_data, batch_size=bs, shuffle=True)
# 输出训练和测试用图片总数
print('训练数据集数量：', len(train_loader.dataset))
print('测试数据集数量：', len(test_loader.dataset))


In [ ]:
# iter将训练数据集读取器转化为可迭代形式
# next获取迭代的下一个元素
# ImageFolder的数据集读取器默认由一组数据共两个组成，即图片本身以及其标签(标签也就是读取文件夹时该图片所在的文件夹)
images, labels = next(iter(train_loader))
print('标签:', labels)
print('图片张量:\n', images)

# 图片张量中
# -1.0000即代表原始的灰度图图片中的0，即黑色
# 1.0000即代表原始的灰度图图片中的255，即白色

In [ ]:
print('图片张量形状:\n', images.shape, '\n             batch_size, 通道数, 长, 宽')

In [ ]:
from matplotlib import pyplot as plt
# 显示前10张图片以及对应标签,检查图片是否是正确的数据集
total = 10 if 10 < bs else bs
for i in range(0, total):
    plt.subplot(2, 5, i + 1)
    # 用images[i]获取该批次中的第i个图片，其是形状为[1, 28, 28]的张量
    # 但plt.imshow接收的数据，通道数应放置在最后，因此使用.permute(1, 2, 0)将其形状变为[28, 28, 1]
    # 设置cmap='gray'以显示灰度图
    plt.imshow(images[i].permute(1, 2, 0), cmap='gray')
    # 设置图片标题
    plt.title('Number: %s' % labels[i].item())
    # 取消坐标轴显示
    plt.xticks([])
    plt.yticks([])
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
# 利用定义类的方式生成网络，pytorch中定义网络需要继承nn.Module
# 在init方法中定义该网络需要的神经网络层
# 在forward方法中梳理神经网络层与层之间的关系
class ForwardNN(nn.Module):
    def __init__(self):
        super(ForwardNN, self).__init__()
        # 卷积层，输入通道数为1，输出通道数为32，卷积核大小为3，步长为1
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        # 卷积层，输入通道数为32，输出通道数为64，卷积核大小为3，步长为1
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        # dropout层，为防止过拟合，以0.25的概率随机将输入张量的某些元素归零
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        # 全连接层，输入为9216
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        # 作为输入，x的形状为[bs, 1, 28, 28]
        x = self.conv1(x)
        # x的形状为[bs, 32, 26, 26]
        # 激活函数relu
        x = F.relu(x)
        x = self.conv2(x)
        # x的形状为[bs, 64, 24, 24]
        x = F.relu(x)
        # max pool，本质上仍然是一种卷积，只不过它会选择卷积核内最大的值作为输出
        x = F.max_pool2d(x, 2)
        # x的形状为[bs, 64, 12, 12]
        x = self.dropout1(x)
        # 将x沿着dim=1的轴平铺开
        x = torch.flatten(x, 1)
        # x的形状为[bs, 1, 9216] (9216 = 64 × 12 × 12)
        x = self.fc1(x)
        # x的形状为[bs, 1, 128]
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        # x的形状为[bs, 1, 10]
        # 激活函数log_softmax
        x = F.log_softmax(x, dim=1)
        # x的形状为[bs, 10]，其取值范围为[-∞, 0)
        return x

In [ ]:
import torch.optim as optim

# 学习率
lr = 0.001
 
model = ForwardNN()
# 定义优化器为Adadelta优化器，并设定学习率
optimizer = optim.Adadelta(model.parameters(), lr=lr)
# 要不要试试Adam优化器？会发现它能够提供十分出色的拟合效果
optimizer = optim.Adam(model.parameters(), lr=lr)

In [ ]:
# 数据集迭代次数，一般迭代越多，越准确，但是当出现过拟合现象后，准确率反而会下降
num_epoch = 3

print("================== Starting Training ==================")
for epoch in range(1, num_epoch + 1):
    # 训练
    model.train()
    for batch_index, (images, labels) in enumerate(train_loader):
        # 初始化优化器
        optimizer.zero_grad()
        # 进行预测
        output = model(images)
        # 使用负对数似然来计算损失值
        # output的值越大，代表着越有可能为该值所在索引的那一类
        # 例如，若其形状为[1, 10]
        # 其值为[-9, -9, -9, -1, -9, -9, -9, -9, -9, -9]
        # 则代表着其最有可能是类别3，并且其余9个类别概率均等
        loss = F.nll_loss(output, labels)
        # 进行梯度的反向传播
        loss.backward()
        # 优化器更新
        optimizer.step()
        # 每训练100批次，进行一次输出
        if batch_index % 100 == 0:
            print('训练 Epoch: {:3d} [{:5d}/{:5d} ({:3d}%)]\t损失值: {:.6f}'.format(epoch, batch_index * len(images), len(train_loader.dataset), 100 * batch_index // len(train_loader), loss.item()))
    # 测试
    model.eval()
    # 记录预测正确的个数
    correct = 0
    # 禁用梯度计算
    with torch.no_grad():
        for batch_index, (images, labels) in enumerate(test_loader):
            output = model(images)
            test_loss = F.nll_loss(output, labels)
            # 选择最大的作为索引作为预测标签
            pred = output.argmax(dim=1, keepdim=True)
            # 统计正确个数并进行累加
            correct += pred.eq(labels.view_as(pred)).sum().item()
    print('\n测试集平均损失: {:.4f}, 准确率: {}/{} ({:.0f}%)\n'.format(test_loss, correct, len(test_loader.dataset), 100. * correct / len(test_loader.dataset)))


In [ ]:
images, labels = next(iter(test_loader))
total = 10 if 10 < bs else bs
for i in range(0, total):
    output = model(images)
    pred = output.argmax(dim=1, keepdim=True)
    print('第'+str(i+1)+'张图片的真实标签为: ', labels[i].item())
    print('第'+str(i+1)+'张图片的预测标签为: ', pred[i].item())
    plt.figure()
    plt.imshow(images[i].permute(1, 2, 0), cmap='gray')
    plt.xticks([])
    plt.yticks([])
    plt.show()